# T-KGAT gộp, không chạy baseline

Input: `tkgat-joint-code.zip` và thư mục **processed/lastfm-3m-k10** đã được Kaggle giải nén.
Không cần thư mục export, raw interactions hay Freebase dump.

Giữ toàn bộ dữ liệu, gamma `{1,3,10}` và 5 seed. Tham số cố định nằm trong `configs/model/tkgat_joint.yaml`.
**Trainer vẫn dùng full graph:** preflight sẽ chặn nếu chỉ riêng graph/embedding đã vượt bộ nhớ. Notebook này không biến trainer thành streaming.

Common settings from Triet: constant lr=0.001, l2=l2_kg=1e-5, dropout=0.1, max_epochs=150, eval_every=5, patience=10.
Attention uses separate train windows in the current code. No absolute tau. Refit on train+validation is enabled by default; relation dates remain endpoint proxies.


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
CODE = Path("/kaggle/working/tkgat-joint-code")
OUTPUT = Path("/kaggle/working/tkgat-joint")
CODE.mkdir(parents=True, exist_ok=True)

archives = list(INPUT.rglob("tkgat-joint-code.zip"))
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            p = Path(name)
            if p.is_absolute() or ".." in p.parts or "\\" in name:
                raise ValueError(f"Unexpected ZIP member: {name}")
        archive.extractall(CODE)
else:
    manifests = list(INPUT.rglob("SOURCE_SHA256.json"))
    if not manifests:
        raise RuntimeError("Add the tkgat-joint-code ZIP/dataset as Input.")
    source = manifests[0].parent
    manifest = json.loads(manifests[0].read_text())
    # A previous notebook output also contains its extracted code. Identical copies are safe.
    if any(json.loads(p.read_text()) != manifest for p in manifests[1:]):
        raise RuntimeError("Input contains different code versions; use the same source snapshot as the saved run.")
    for name in [*manifest, "SOURCE_SHA256.json"]:
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts or "\\" in name:
            raise ValueError(f"Unexpected source path: {name}")
        target = CODE / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source / relative, target)

manifest = json.loads((CODE / "SOURCE_SHA256.json").read_text())
for name, expected in manifest.items():
    if hashlib.sha256((CODE / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError(f"Source hash mismatch: {name}")

datasets = [
    p.parent
    for p in INPUT.rglob("meta.json")
    if p.parent.name in {"lastfm-3m-k10", "lastfm-raw"}
    and (p.parent / "train.csv").is_file()
    and (p.parent / "kg/default/kg_meta.json").is_file()
]
if len(datasets) != 1:
    raise RuntimeError(
        "Add exactly one extracted processed/lastfm-3m-k10 directory as Input; CSV files must be visible."
    )
PROCESSED = datasets[0]

In [ ]:
# Preserve Kaggle's CUDA PyTorch. Detailed output remains in tkgat-joint/training.log.
import ast
import re
from collections import deque

import yaml

MODEL_CONFIG = CODE / "configs/model/tkgat_joint.yaml"
params = yaml.safe_load(MODEL_CONFIG.read_text())


def run_tkgat(*extra):
    command = [
        sys.executable,
        "-u",
        str(CODE / "scripts/kaggle_tkgat_joint.py"),
        "--processed",
        str(PROCESSED),
        "--out",
        str(OUTPUT),
        *map(str, extra),
    ]
    env = {**os.environ, "PYTHONPATH": str(CODE / "src")}
    seeds = params["seeds"]
    gammas = [point["gamma"] for point in params["grid"]]
    selection_count = len(seeds) * len(gammas)
    refit = "--no-refit" not in extra and "--val-only" not in extra
    total = selection_count + (len(seeds) if refit else 0)
    selected_gamma = None
    completed = set()
    active = None
    quiet_check = "--check-only" in extra or "--benchmark-batches" in extra
    tail = deque(maxlen=60)

    def fit_name(phase, seed, gamma):
        number = (
            selection_count + seeds.index(seed) + 1
            if phase == "refit"
            else gammas.index(gamma) * len(seeds) + seeds.index(seed) + 1
        )
        return number, f"run {number}/{total} ({phase}, gamma={gamma:g}, seed={seed})"

    with subprocess.Popen(
        command,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    ) as process:
        for line in process.stdout:
            tail.append(line)
            if quiet_check:
                continue
            progress = re.match(r"\[train:(select|refit) seed=(\d+) gamma=([^\]]+)\] (.*)", line)
            result = re.match(r"\[val\] grid=(\{.*\}) seed=(\d+)", line)
            test = re.match(r"\[test\] seed=(\d+)", line)
            chosen = re.match(r"selected grid point (\{.*\}) ", line)
            if progress:
                phase, seed, gamma, message = progress.groups()
                active = fit_name(phase, int(seed), float(gamma))
                if message.startswith("Initializing"):
                    print(f"Running {active[1]}", flush=True)
                elif message.startswith("Resumed after epoch"):
                    print(f"Continuing {active[1]}: {message.strip()}", flush=True)
            elif result:
                point, seed = result.groups()
                number, name = fit_name("select", int(seed), float(ast.literal_eval(point)["gamma"]))
                if number not in completed:
                    print(f"Result ready: {name}", flush=True)
                    completed.add(number)
            elif chosen:
                selected_gamma = float(ast.literal_eval(chosen[1])["gamma"])
            elif test and refit:
                number, name = fit_name("refit", int(test[1]), selected_gamma)
                if number not in completed:
                    print(f"Result ready: {name}", flush=True)
                    completed.add(number)
            elif line.startswith(("[pause]", "[pause:refit]")):
                saved = re.search(r"saved epoch (\d+)", line)
                suffix = f" after epoch {saved[1]}" if saved else ""
                print(f"Saved {active[1]}{suffix}; continue this run next session.", flush=True)
            elif line.startswith(("Partial grid saved.", "[resume] Fit budget reached")):
                next_number = next((i for i in range(1, total + 1) if i not in completed), None)
                if next_number is not None:
                    print(f"Session saved. Next: run {next_number}/{total}.", flush=True)
            elif line.startswith("test mean"):
                print("All runs complete.", flush=True)
        returncode = process.wait()
    if returncode:
        print("".join(tail), end="", flush=True)
        raise subprocess.CalledProcessError(returncode, command)
    if not quiet_check:
        print(f"Results directory: {OUTPUT}", flush=True)


run_tkgat("--check-only")
report = json.loads((OUTPUT / "memory-report.json").read_text())
if report["blockers"]:
    print("\n".join(report["blockers"]))

In [ ]:
# Run this small timing check before committing to full training.
RUN_BENCHMARK = True
BENCHMARK_BATCHES = 5
RUN_REFIT = True
RESTORE_FROM = None  # A saved tkgat-joint output directory from a compatible previous session.
benchmark = None


def has_training_progress():
    return (OUTPUT / "registry.csv").exists() or any((OUTPUT / "runs").rglob("*.pt"))


if RUN_BENCHMARK:
    extra = ["--benchmark-batches", str(BENCHMARK_BATCHES), "--refit" if RUN_REFIT else "--no-refit"]
    if RESTORE_FROM and not has_training_progress():
        extra += ["--restore", RESTORE_FROM]
    run_tkgat(*extra)
    benchmark_file = OUTPUT / "runs" / f"{PROCESSED.name}-joint" / "tkgat_joint_benchmark.json"
    benchmark = json.loads(benchmark_file.read_text())
    seconds = benchmark["estimated_epoch_seconds_excluding_validation"]
    print(f"Estimated epoch: {seconds / 60:.1f} minutes, excluding validation/checkpoint overhead.")

In [ ]:
RUN_TRAINING = False  # Benchmark first, then enable training.
MAX_NEW_RUNS = 2  # Stop after at most two completed selection/refit fits.
EPOCHS_PER_FIT = None  # Complete each fit; checkpoints are still saved after every epoch.

if RUN_TRAINING:
    if report["blockers"]:
        raise RuntimeError(
            "Preflight blocked training: full graph requires larger hardware or a different memory backend."
        )
    if benchmark is None:
        raise RuntimeError("Run the benchmark cell first to estimate epoch duration.")
    extra = [
        "--max-new-runs",
        str(MAX_NEW_RUNS),
        "--refit" if RUN_REFIT else "--no-refit",
    ]
    if EPOCHS_PER_FIT is not None:
        extra += ["--epochs-per-fit", str(EPOCHS_PER_FIT)]
    if RESTORE_FROM and not has_training_progress():
        extra += ["--restore", RESTORE_FROM]
    run_tkgat(*extra)
else:
    print("Training is off. Benchmark timing is not an official model fit.")

## Lưu và tiếp tục

Phiên đầu: Input chỉ cần code và processed, `RESTORE_FROM=None`. Phiên sau giữ nguyên hai Input đó, thêm output mới nhất của phiên trước và đặt `RESTORE_FROM` tới thư mục `tkgat-joint` bên trong. Không cần upload lại code/data; chỉ dùng bản result mới nhất vì nó đã chứa tiến độ các phiên trước.
Lưu phiên bản notebook **cùng output**, giữ toàn bộ `/kaggle/working/tkgat-joint/` (registry, runs/checkpoint, identity, log).
Mỗi lần chạy ô train hoàn tất tối đa 2 lượt gamma/seed hoặc refit/seed rồi dừng. Giới hạn 2 lượt không đảm bảo kịp 12 giờ; checkpoint vẫn lưu mỗi epoch nếu lượt đang chạy bị ngắt.
Phiên sau thêm output đã lưu làm Input, đặt `RESTORE_FROM` tới thư mục `tkgat-joint`, rồi chạy lại notebook.
Trong cùng phiên chỉ cần chạy lại ô train. Checkpoint lưu sau mỗi epoch; ngắt giữa epoch chỉ chạy lại epoch chưa lưu. Giữ cùng code/config và loại thiết bị khi tiếp tục.
Chỉ sau khi đủ **3 gamma × 5 seed**, runner chọn gamma trên validation rồi chấm test của gamma đã chọn.
Không dùng checkpoint Stage 1/2 hoặc baseline cũ.
